In [18]:
import json
import os
import pandas as pd
from shapely.geometry import shape
from shapely.wkt import dumps as dumps_wkt
from google.cloud import bigquery
from google.oauth2 import service_account

def upload_geojson_to_bigquery(credentials_path, project_id, dataset_id, geojson_table_pairs):
    # Create BigQuery client
    credentials = service_account.Credentials.from_service_account_file(credentials_path)
    client = bigquery.Client(credentials=credentials, project=project_id)

    for geojson_path, target_table_id in geojson_table_pairs:
        print(f"Processing GeoJSON file: {geojson_path}")
        print(f"Target BigQuery table: {target_table_id}")

        # Load GeoJSON data from file
        with open(geojson_path) as f:
            geojson_data = json.load(f)
        
        print("GeoJSON data loaded successfully.")

        # Extract features and properties
        data = []
        for feature in geojson_data['features']:
            properties = feature['properties']
            geometry = shape(feature['geometry'])
            properties['geometry'] = dumps_wkt(geometry)
            data.append(properties)

        print(f"Extracted {len(data)} features from GeoJSON data.")

        # Convert to DataFrame
        df = pd.DataFrame(data)
        print("Data converted to DataFrame.")

        # Define the fully qualified table ID in BigQuery
        full_table_id = f'{project_id}.{dataset_id}.{target_table_id}'
        print(f"Full table ID: {full_table_id}")

        # Dynamically infer schema from DataFrame and adjust for GEOGRAPHY type
        schema = []
        for column in df.columns:
            if column == 'geometry':
                schema.append(bigquery.SchemaField(column, "GEOGRAPHY"))
            else:
                schema.append(bigquery.SchemaField(column, "STRING"))
        print(f"Schema inferred: {[field.name + ' ' + field.field_type for field in schema]}")

        # Delete the existing table if it exists
        try:
            print(f"Deleting existing table {full_table_id} if it exists...")
            client.delete_table(full_table_id)
            print("Table deleted successfully.")
        except Exception as e:
            print(f"No existing table to delete or an error occurred: {e}")

        # Create or replace table with the correct schema
        create_table_query = f"""
        CREATE TABLE `{full_table_id}` (
            {', '.join([f"{field.name} {field.field_type}" for field in schema])}
        )
        """
        print("Creating table with the following schema...")
        print(create_table_query)
        client.query(create_table_query).result()
        print("Table created successfully.")

        # Convert DataFrame to list of dictionaries
        records = df.to_dict(orient='records')
        print(f"Data converted to list of dictionaries with {len(records)} records.")

        # Load data into BigQuery table
        print("Starting data load into BigQuery...")
        load_job = client.load_table_from_json(
            records,
            full_table_id,
            job_config=bigquery.LoadJobConfig(schema=schema)
        )

        # Wait for the job to complete
        load_job.result()
        print(f'Loaded {load_job.output_rows} rows into {full_table_id}.')

# Example usage
credentials_path = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS_PATH')  # Update with your credentials file path
project_id = 'personal-nurfaldi'
dataset_id = 'geojson_data'

# Define a list of GeoJSON file paths and their corresponding target table IDs
geojson_table_pairs = [
    # ('data_2/geojson/Sulawesi Tenggara_DESA_KELURAHAN.geojson', 'sultra_kelurahan'),
    # ('data_2/geojson/Sulawesi Tenggara_KECAMATAN.geojson', 'sultra_kecamatan'),
    ('data_2/geojson/Sulawesi Tenggara_KAB_KOTA_try.geojson', 'sultra_kabupaten'),
    # ('DKI.geojson', 'dki_kelurahan')
]

# Call the function to upload the filtered GeoJSON data to BigQuery
upload_geojson_to_bigquery(credentials_path, project_id, dataset_id, geojson_table_pairs)


Processing GeoJSON file: data_2/geojson/Sulawesi Tenggara_KAB_KOTA_try.geojson
Target BigQuery table: sultra_kabupaten
GeoJSON data loaded successfully.
Extracted 17 features from GeoJSON data.
Data converted to DataFrame.
Full table ID: personal-nurfaldi.geojson_data.sultra_kabupaten
Schema inferred: ['KODE_KAB_KOTA STRING', 'KODE_PROVINSI STRING', 'NAMA_KAB_KOTA STRING', 'NAMA_PROVINSI STRING', 'LUAS_HA STRING', 'LUAS_SQKM STRING', 'KODE STRING', 'geometry GEOGRAPHY']
Deleting existing table personal-nurfaldi.geojson_data.sultra_kabupaten if it exists...
Table deleted successfully.
Creating table with the following schema...

        CREATE TABLE `personal-nurfaldi.geojson_data.sultra_kabupaten` (
            KODE_KAB_KOTA STRING, KODE_PROVINSI STRING, NAMA_KAB_KOTA STRING, NAMA_PROVINSI STRING, LUAS_HA STRING, LUAS_SQKM STRING, KODE STRING, geometry GEOGRAPHY
        )
        
Table created successfully.
Data converted to list of dictionaries with 17 records.
Starting data load int

In [8]:
import json
import os
import pandas as pd
from shapely.geometry import shape, mapping
from shapely.wkt import dumps as dumps_wkt
from google.cloud import bigquery
from google.oauth2 import service_account
from pyproj import Transformer

def convert_to_lat_lon(coords):
    # Define the transformer based on your original CRS (coordinate reference system) and WGS84
    # For example, if your original CRS is EPSG:3857 (Web Mercator)
    transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    return [transformer.transform(x, y) for x, y in coords]

def convert_geometry(geometry):
    if geometry['type'] == 'Polygon':
        coords = geometry['coordinates'][0]
        lat_lon_coords = convert_to_lat_lon(coords)
        return {
            'type': 'Polygon',
            'coordinates': [lat_lon_coords]
        }
    elif geometry['type'] == 'MultiPolygon':
        coords = [convert_to_lat_lon(polygon[0]) for polygon in geometry['coordinates']]
        return {
            'type': 'MultiPolygon',
            'coordinates': coords
        }
    else:
        # Handle other geometry types if necessary
        return geometry

def upload_geojson_to_bigquery(credentials_path, project_id, dataset_id, geojson_table_pairs):
    # Create BigQuery client
    credentials = service_account.Credentials.from_service_account_file(credentials_path)
    client = bigquery.Client(credentials=credentials, project=project_id)

    for geojson_path, target_table_id in geojson_table_pairs:
        print(f"Processing GeoJSON file: {geojson_path}")
        print(f"Target BigQuery table: {target_table_id}")

        # Load GeoJSON data from file
        with open(geojson_path) as f:
            geojson_data = json.load(f)

        print("GeoJSON data loaded successfully.")

        # Extract features and properties
        data = []
        for feature in geojson_data['features']:
            properties = feature['properties']
            geometry = convert_geometry(feature['geometry'])
            properties['geometry'] = dumps_wkt(shape(geometry))
            data.append(properties)

        print(f"Extracted {len(data)} valid features from GeoJSON data.")

        # Convert to DataFrame
        df = pd.DataFrame(data)
        print("Data converted to DataFrame.")

        # Define the fully qualified table ID in BigQuery
        full_table_id = f'{project_id}.{dataset_id}.{target_table_id}'
        print(f"Full table ID: {full_table_id}")

        # Dynamically infer schema from DataFrame and adjust for GEOGRAPHY type
        schema = []
        for column in df.columns:
            if column == 'geometry':
                schema.append(bigquery.SchemaField(column, "GEOGRAPHY"))
            else:
                schema.append(bigquery.SchemaField(column, "STRING"))
        print(f"Schema inferred: {[field.name + ' ' + field.field_type for field in schema]}")

        # Delete the existing table if it exists
        try:
            print(f"Deleting existing table {full_table_id} if it exists...")
            client.delete_table(full_table_id)
            print("Table deleted successfully.")
        except Exception as e:
            print(f"No existing table to delete or an error occurred: {e}")

        # Create or replace table with the correct schema
        create_table_query = f"""
        CREATE TABLE `{full_table_id}` (
            {', '.join([f"{field.name} {field.field_type}" for field in schema])}
        )
        """
        print("Creating table with the following schema...")
        print(create_table_query)
        client.query(create_table_query).result()
        print("Table created successfully.")

        # Convert DataFrame to list of dictionaries
        records = df.to_dict(orient='records')
        print(f"Data converted to list of dictionaries with {len(records)} records.")

        # Load data into BigQuery table
        print("Starting data load into BigQuery...")
        load_job = client.load_table_from_json(
            records,
            full_table_id,
            job_config=bigquery.LoadJobConfig(schema=schema)
        )

        # Wait for the job to complete
        load_job.result()
        print(f'Loaded {load_job.output_rows} rows into {full_table_id}.')

# Example usage
credentials_path = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS_PATH')  # Update with your credentials file path
project_id = 'personal-nurfaldi'
dataset_id = 'geojson_data'

# Define a list of GeoJSON file paths and their corresponding target table IDs
geojson_table_pairs = [
    ('data_2/geojson/Sulawesi Tenggara_DESA_KELURAHAN.geojson', 'sultra_kelurahan'),
    ('data_2/geojson/Sulawesi Tenggara_KECAMATAN.geojson', 'sultra_kecamatan'),
    ('data_2/geojson/Sulawesi Tenggara_KAB_KOTA.geojson', 'sultra_kabupaten'),
    ('DKI.geojson', 'dki_kelurahan')
]

# Call the function to upload the filtered GeoJSON data to BigQuery
upload_geojson_to_bigquery(credentials_path, project_id, dataset_id, geojson_table_pairs)


Processing GeoJSON file: data_2/geojson/Sulawesi Tenggara_KAB_KOTA_cleaned.geojson
Target BigQuery table: sultra_kabupaten
GeoJSON data loaded successfully.


TypeError: 'float' object is not iterable

In [22]:
output_file = 'data_2/geojson/Sulawesi Tenggara_KAB_KOTA_try.geojson'
input_file = 'data_2/geojson/Sulawesi Tenggara_KAB_KOTA.geojson'

import geopandas as gpd
from shapely.geometry import shape
from shapely.ops import transform
from pyproj import Transformer

# Define a function to remove Z elevation
def remove_z(geometry):
    if geometry.has_z:
        # Remove Z elevation
        return transform(lambda x, y, z=None: (x, y), geometry)
    return geometry

# Define the CRS transformers
transformer = Transformer.from_crs("EPSG:32749", "EPSG:4326")

# Load the GeoJSON file
# input_file = 'input.geojson'
gdf = gpd.read_file(input_file)

# Check the current CRS
print(f"Current CRS: {gdf.crs}")

# Reproject to EPSG:4326
gdf = gdf.to_crs(epsg=4326)

# Remove Z elevation
gdf['geometry'] = gdf['geometry'].apply(remove_z)

# Save the reprojected and Z-free GeoJSON to a new file
# output_file = 'output_reprojected_no_z.geojson'
gdf.to_file(output_file, driver='GeoJSON')

print(f"Reprojected and Z-free GeoJSON saved to: {output_file}")



Current CRS: EPSG:32749


TypeError: 'MultiPolygon' object is not iterable

In [20]:
gdf

,KODE_KAB_KOTA,KODE_PROVINSI,NAMA_KAB_KOTA,NAMA_PROVINSI,LUAS_HA,LUAS_SQKM,KODE,geometry
0,74.01,74,Kolaka,Sulawesi Tenggara,305912.139908,3059.121399,74,"MULTIPOLYGON (((121.36979 -4.08885, 121.37599 ..."
1,74.02,74,Konawe,Sulawesi Tenggara,555096.405594,5550.964056,74,"MULTIPOLYGON (((121.95259 -4.03344, 121.95442 ..."
2,74.03,74,Muna,Sulawesi Tenggara,193418.196282,1934.181963,74,"MULTIPOLYGON (((122.50441 -5.20850, 122.49755 ..."
3,74.04,74,Buton,Sulawesi Tenggara,174124.077234,1741.240772,74,"MULTIPOLYGON (((122.85157 -5.61742, 122.83752 ..."
4,74.05,74,Konawe Selatan,Sulawesi Tenggara,440529.573690,4405.295737,74,"MULTIPOLYGON (((122.12930 -4.53031, 122.12597 ..."
5,74.06,74,Bombana,Sulawesi Tenggara,341278.050677,3412.780507,74,"MULTIPOLYGON (((121.50820 -4.83824, 121.50916 ..."
6,74.07,74,Wakatobi,Sulawesi Tenggara,47626.604584,476.266046,74,"MULTIPOLYGON (((123.58263 -5.38470, 123.58285 ..."
7,74.08,74,Kolaka Utara,Sulawesi Tenggara,302501.902732,3025.019027,74,"MULTIPOLYGON (((120.99665 -3.65910, 120.98473 ..."
8,74.09,74,Konawe Utara,Sulawesi Tenggara,437743.458439,4377.434584,74,"MULTIPOLYGON (((122.06984 -3.63232, 122.06542 ..."
9,74.10,74,Buton Utara,Sulawesi Tenggara,183269.373881,1832.693739,74,"MULTIPOLYGON (((122.96679 -5.12356, 122.96507 ..."
